# Model deep-dive: SLA breach risk (baseline)

Goal: predict whether a support ticket (**Case**) will breach its SLA.

This is intentionally a **simple, interpretable baseline**:
- One-hot encoding for categorical fields (priority, channel, case_type, etc.)
- Logistic regression with class balancing
- Metrics: ROC AUC, Average Precision, Accuracy, Precision/Recall/F1

## Why this matters in Dynamics 365 / Dataverse
In a real deployment you would export Cases from Dataverse, run this model, and surface risk scores back into Dynamics as a custom field — enabling proactive SLA management.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent
df = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'case_features.csv')
print(df.shape)
df.head()

In [ ]:
target = 'sla_breached'
feature_cols = [
    'priority','channel','case_type',
    'country','segment','tier',
    'employees','tenure_months',
    'created_dow','created_hour',
    'sla_hours'
]

X = df[feature_cols].copy()
y = df[target].astype(int)

cat_cols = ['priority','channel','case_type','country','segment','tier']
num_cols = [c for c in feature_cols if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

pipe = Pipeline([
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=500, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('ROC AUC :', round(roc_auc_score(y_test, proba), 4))
print('Avg Precision:', round(average_precision_score(y_test, proba), 4))
print()
print(classification_report(y_test, pred, digits=3))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
ax.set_title('Confusion Matrix – SLA breach risk')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (log-reg coefficients for top OHE features)
import numpy as np

ohe = pipe.named_steps['pre'].named_transformers_['cat']
ohe_names = ohe.get_feature_names_out(cat_cols).tolist()
all_names = ohe_names + num_cols

coefs = pipe.named_steps['clf'].coef_[0]
feat_imp = pd.Series(coefs, index=all_names).sort_values(key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.sort_values().plot(kind='barh', ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 logistic regression coefficients')
ax.set_xlabel('Coefficient (log-odds)')
plt.tight_layout()
plt.show()